In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

SEED = 0

# 데이터 로드
train_df = pd.read_csv(
    '../data/santander-customer-satisfaction/train.csv'
)

test_df = pd.read_csv(
    '../data/santander-customer-satisfaction/test.csv'
)


In [3]:
y_label = train_df['TARGET']

In [4]:
# ID, TARGET 제거
X_feature = train_df.drop(columns=['ID', 'TARGET'])
X_test = test_df.drop(columns=['ID']).copy()

# 나중의 후처리 규칙 확인용 원본
X_train_c = X_feature.copy()

# 행마다 값이 0인 Feature 개수
X_feature['n0'] = (X_feature == 0).sum(axis=1)


C:\Users\mega\AppData\Local\Temp\ipykernel_2812\2605123168.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_feature['n0'] = (X_feature == 0).sum(axis=1)


In [5]:
# 상수 Feature 찾기
constant_cols = [
    col for col in X_feature.columns
    if X_feature[col].nunique() == 1
]

print("상수 Feature:", len(constant_cols))

# 상수 Feature 제거
X_feature = X_feature.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)


상수 Feature: 34


In [6]:
# 완전히 동일한 중복 Feature 찾기
duplicate_cols = X_feature.columns[
    X_feature.T.duplicated()
].tolist()

print("중복 Feature:", len(duplicate_cols))

# 중복 Feature 제거
X_feature = X_feature.drop(columns=duplicate_cols)
X_test = X_test.drop(columns=duplicate_cols)
print("최종 Feature 개수:", X_feature.shape[1])

중복 Feature: 29
최종 Feature 개수: 307


In [7]:
#var3 이상치 제거
X_feature= X_feature.replace(-999999,2)
#var38 높은 수치 제거
X_feature['var38'] = np.log(X_feature['var38'])

In [8]:
#학습/테스트 데이터 분리, 분포 확인
from sklearn.model_selection import train_test_split




X_train, X_test, y_train, y_test = train_test_split(X_feature, y_label,
                                                    test_size=0.2, random_state=0, stratify=y_label)
train_cnt = y_train.count()
test_cnt = y_test.count()
print('학습 세트 Shape:{0}, 테스트 세트 Shape:{1}'.format(X_train.shape , X_test.shape))




print(' 학습 세트 레이블 값 분포 비율')
print(y_train.value_counts()/train_cnt)
print('\n 테스트 세트 레이블 값 분포 비율')
print(y_test.value_counts()/test_cnt)


학습 세트 Shape:(60816, 307), 테스트 세트 Shape:(15204, 307)
 학습 세트 레이블 값 분포 비율
TARGET
0    0.960438
1    0.039562
Name: count, dtype: float64

 테스트 세트 레이블 값 분포 비율
TARGET
0    0.960405
1    0.039595
Name: count, dtype: float64


In [9]:
# X_train, y_train을 다시 학습과 검증 데이터 세트로 분리.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train,
                                                    test_size=0.3, random_state=0, stratify=y_train)


In [10]:
# XGB모델 학습 AUC 점수 확인
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score




# n_estimators는 500으로, learning_rate 0.05, random state는 예제 수행 시마다 동일 예측 결과를 위해 설정.
xgb_clf = XGBClassifier(n_estimators=500, learning_rate=0.05, early_stopping_rounds=100, eval_metric='auc',random_state=156)


In [11]:
# 성능 평가 지표를 auc로, 조기 중단 파라미터는 100으로 설정하고 학습 수행.
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)])




xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(xgb_roc_score))


[0]	validation_0-auc:0.84533	validation_1-auc:0.82264
[1]	validation_0-auc:0.85296	validation_1-auc:0.82895
[2]	validation_0-auc:0.85349	validation_1-auc:0.82858
[3]	validation_0-auc:0.85571	validation_1-auc:0.83482
[4]	validation_0-auc:0.85741	validation_1-auc:0.83567
[5]	validation_0-auc:0.85777	validation_1-auc:0.83682
[6]	validation_0-auc:0.85802	validation_1-auc:0.83728
[7]	validation_0-auc:0.85844	validation_1-auc:0.83731
[8]	validation_0-auc:0.85891	validation_1-auc:0.83713
[9]	validation_0-auc:0.86162	validation_1-auc:0.83886
[10]	validation_0-auc:0.86335	validation_1-auc:0.83967
[11]	validation_0-auc:0.86485	validation_1-auc:0.84009
[12]	validation_0-auc:0.86558	validation_1-auc:0.83996
[13]	validation_0-auc:0.86649	validation_1-auc:0.84116
[14]	validation_0-auc:0.86736	validation_1-auc:0.84134
[15]	validation_0-auc:0.86804	validation_1-auc:0.84108
[16]	validation_0-auc:0.86848	validation_1-auc:0.84046
[17]	validation_0-auc:0.86957	validation_1-auc:0.84111
[18]	validation_0-au

In [12]:
#feaure_importance 시행하기
feature_importance = pd.Series(
    xgb_clf.feature_importances_,
    index=X_train.columns
)

# 중요도가 0인 Feature 제거
feature_importance = feature_importance[feature_importance > 0]

# 중요도가 높은 순서대로 정렬
feature_importance = feature_importance.sort_values(ascending=False)

print(feature_importance)

X_feature = X_feature.loc[:, xgb_clf.feature_importances_ != 0]

saldo_var30                0.118671
var15                      0.055953
saldo_var8                 0.054150
num_var42_0                0.021599
num_meses_var5_ult3        0.019886
                             ...   
saldo_var1                 0.001210
num_op_var40_ult1          0.001090
imp_op_var40_comer_ult1    0.000875
saldo_var12                0.000861
num_op_var40_comer_ult1    0.000836
Length: 118, dtype: float32


In [13]:
X_feature.describe()

,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var41_comer_ult1,imp_op_var41_comer_ult3,imp_op_var41_efect_ult1,...,saldo_medio_var12_hace2,saldo_medio_var12_hace3,saldo_medio_var12_ult1,saldo_medio_var12_ult3,saldo_medio_var13_corto_hace2,saldo_medio_var13_corto_hace3,saldo_medio_var13_corto_ult1,saldo_medio_var13_corto_ult3,var38,n0
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,7.602000e+04,76020.000000,7.602000e+04,7.602000e+04,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000
mean,2.716483,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,68.803937,113.056934,68.205140,...,3.997023e+03,613.534443,5.703008e+03,4.401002e+03,3639.419939,556.184178,4852.261814,3857.848542,11.482248,335.426888
std,9.447971,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,319.605516,512.154823,531.897917,...,3.777314e+04,9292.752726,4.620254e+04,3.550718e+04,26359.174223,7182.642532,31886.615189,25572.245055,0.560589,17.836658
min,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,8.549418,220.000000
25%,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,11.125358,325.000000
50%,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,11.575047,340.000000
75%,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,11.684828,348.000000
max,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,12888.030000,16566.810000,45990.000000,...,3.000538e+06,668335.320000,3.004186e+06,2.272859e+06,450000.000000,304838.700000,450000.000000,450000.000000,16.908131,361.000000


In [14]:
# log_features = X_feature.columns[(X_feature >= 1000).any()]

# X_feature[log_features] = np.log1p(X_feature[log_features])

In [15]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# PCA에 사용할 feature
features = X_feature.columns[1:-1]

# 1000 이상의 값이 존재하는 feature
log_features = X_feature[features].columns[
    (X_feature[features].max() >= 1000) &
    (X_feature[features].min() >= 0)
]

print("로그 변환 대상:", log_features.tolist())

# 로그 변환
X_feature[log_features] = np.log1p(X_feature[log_features])

# 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_feature[features])

# PCA
pca = PCA(n_components=2)
x_train_projected = pca.fit_transform(X_scaled)

# PCA 결과 추가
X_feature.insert(1, 'PCAOne', x_train_projected[:, 0])
X_feature.insert(2, 'PCATwo', x_train_projected[:, 1])

로그 변환 대상: ['imp_ent_var16_ult1', 'imp_op_var39_comer_ult1', 'imp_op_var39_comer_ult3', 'imp_op_var40_comer_ult1', 'imp_op_var40_comer_ult3', 'imp_op_var41_comer_ult1', 'imp_op_var41_comer_ult3', 'imp_op_var41_efect_ult1', 'imp_op_var41_efect_ult3', 'imp_op_var41_ult1', 'imp_op_var39_efect_ult1', 'imp_op_var39_efect_ult3', 'imp_op_var39_ult1', 'imp_sal_var16_ult1', 'saldo_var12', 'saldo_var13_corto', 'saldo_var13', 'saldo_var24', 'saldo_var26', 'saldo_var25', 'saldo_var37', 'imp_aport_var13_hace3', 'imp_var43_emit_ult1', 'imp_trans_var37_ult1', 'saldo_medio_var8_hace3', 'saldo_medio_var12_hace2', 'saldo_medio_var12_hace3', 'saldo_medio_var12_ult1', 'saldo_medio_var12_ult3', 'saldo_medio_var13_corto_hace2', 'saldo_medio_var13_corto_hace3', 'saldo_medio_var13_corto_ult1', 'saldo_medio_var13_corto_ult3']


C:\Users\mega\AppData\Local\Temp\ipykernel_2812\3483345028.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_feature.insert(1, 'PCAOne', x_train_projected[:, 0])
C:\Users\mega\AppData\Local\Temp\ipykernel_2812\3483345028.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_feature.insert(2, 'PCATwo', x_train_projected[:, 1])


In [16]:
# from sklearn.preprocessing import normalize
# from sklearn.decomposition import PCA

# pca = PCA(n_components=2)
# features=X_feature.columns[1:-1]
# x_train_projected = pca.fit_transform(normalize(X_feature[features], axis=0))
# X_feature.insert(1, 'PCAOne', x_train_projected[:, 0])
# X_feature.insert(1, 'PCATwo', x_train_projected[:, 1])


In [17]:
#재학습

X_train, X_test, y_train, y_test = train_test_split(X_feature, y_label,
                                                    test_size=0.2, random_state=0, stratify=y_label)
train_cnt = y_train.count()
test_cnt = y_test.count()
print('학습 세트 Shape:{0}, 테스트 세트 Shape:{1}'.format(X_train.shape , X_test.shape))




print(' 학습 세트 레이블 값 분포 비율')
print(y_train.value_counts()/train_cnt)
print('\n 테스트 세트 레이블 값 분포 비율')
print(y_test.value_counts()/test_cnt)

학습 세트 Shape:(60816, 120), 테스트 세트 Shape:(15204, 120)
 학습 세트 레이블 값 분포 비율
TARGET
0    0.960438
1    0.039562
Name: count, dtype: float64

 테스트 세트 레이블 값 분포 비율
TARGET
0    0.960405
1    0.039595
Name: count, dtype: float64


In [18]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train,
                                                    test_size=0.3, random_state=0, stratify=y_train)


In [19]:
# XGB모델 학습 AUC 점수 확인
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score




# n_estimators는 500으로, learning_rate 0.05, random state는 예제 수행 시마다 동일 예측 결과를 위해 설정.
xgb_clf = XGBClassifier(n_estimators=500, learning_rate=0.05, early_stopping_rounds=100, eval_metric='auc',random_state=156)

In [20]:
# 성능 평가 지표를 auc로, 조기 중단 파라미터는 100으로 설정하고 학습 수행.
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)])




xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(xgb_roc_score))


[0]	validation_0-auc:0.84652	validation_1-auc:0.82181
[1]	validation_0-auc:0.85320	validation_1-auc:0.82847
[2]	validation_0-auc:0.85427	validation_1-auc:0.82886
[3]	validation_0-auc:0.85655	validation_1-auc:0.83405
[4]	validation_0-auc:0.85776	validation_1-auc:0.83497
[5]	validation_0-auc:0.85826	validation_1-auc:0.83660
[6]	validation_0-auc:0.85928	validation_1-auc:0.83704
[7]	validation_0-auc:0.85962	validation_1-auc:0.83692
[8]	validation_0-auc:0.86271	validation_1-auc:0.83928
[9]	validation_0-auc:0.86475	validation_1-auc:0.84017
[10]	validation_0-auc:0.86570	validation_1-auc:0.84105
[11]	validation_0-auc:0.86668	validation_1-auc:0.84183
[12]	validation_0-auc:0.86740	validation_1-auc:0.84202
[13]	validation_0-auc:0.86805	validation_1-auc:0.84225
[14]	validation_0-auc:0.86855	validation_1-auc:0.84260
[15]	validation_0-auc:0.86949	validation_1-auc:0.84230
[16]	validation_0-auc:0.87000	validation_1-auc:0.84246
[17]	validation_0-auc:0.87019	validation_1-auc:0.84285
[18]	validation_0-au